# AIMO 3 Submission: SC-TIR with Transformers

**Algorithm: Self-Consistency with Tool-Integrated Reasoning**

```
For each problem:
  1. Generate N=16 solution attempts (with temperature sampling)
  2. Each attempt can write Python code -> we execute it -> feed output back
  3. Repeat code execution up to M=4 times per attempt
  4. Extract \boxed{answer} from each attempt
  5. Majority vote -> final answer
```

**IMPORTANT SETUP (internet is OFF):**
1. Click **"Add Input"** (right sidebar) -> **"Models"** -> search **"qwen2.5 math"**
2. Select **Qwen2.5-Math-7B-Instruct** (by QwenLM) -> Add
3. Also add the competition data: **"Add Input"** -> **"Competitions"** -> **"AIMO Progress Prize 3"**
4. Set accelerator to **GPU** (H100 preferred)

No pip installs needed - uses only pre-installed packages (transformers, torch).

In [ ]:
# No pip install needed! transformers is pre-installed on Kaggle.
# We try vLLM first (faster), fall back to transformers (always works).

USE_VLLM = False
try:
    from vllm import LLM, SamplingParams
    USE_VLLM = True
    print("vLLM found - using fast batched inference")
except ImportError:
    print("Using transformers backend (pre-installed, no internet needed)")

print(f"Backend: {'vLLM' if USE_VLLM else 'transformers'}")

In [ ]:
import os
import re
import signal
import subprocess
import tempfile
from collections import Counter
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Optional

import pandas as pd
import torch

IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

print(f"Submission mode: {IS_SUBMISSION}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# =============================================
# CONFIGURATION
# =============================================

@dataclass
class Config:
    num_samples: int = 16        # N: solution attempts per problem (16 for transformers speed)
    num_generations: int = 4     # M: code execution rounds per attempt
    temperature: float = 0.7     # Higher = more diverse solutions
    max_tokens: int = 2048       # Max tokens per generation step
    code_timeout: int = 5        # Seconds before killing code execution
    model_path: str = ""         # Set below

config = Config()

# ---- Find the model ----
# With internet OFF, model MUST be attached as Kaggle input.
# Go to: Add Input -> Models -> search "qwen2.5 math" -> add 7b-instruct
import glob

SEARCH_PATHS = [
    # Common Kaggle model attachment paths
    "/kaggle/input/qwen2.5-math/transformers/7b-instruct/1",
    "/kaggle/input/qwen2.5-math/transformers/7b-instruct/2",
    "/kaggle/input/qwen-2.5-math/transformers/7b-instruct/1",
    "/kaggle/input/qwen2.5-math-7b-instruct",
    # Custom trained model
    "/kaggle/input/aimo3-trained-model/final_model",
]

# Also search any /kaggle/input/ directory that looks like qwen
for p in glob.glob("/kaggle/input/*/transformers/*/"):
    SEARCH_PATHS.append(p.rstrip("/"))
for p in glob.glob("/kaggle/input/*/"):
    if "qwen" in p.lower() or "math" in p.lower():
        SEARCH_PATHS.append(p.rstrip("/"))

for path in SEARCH_PATHS:
    if os.path.exists(path) and (os.path.isfile(os.path.join(path, "config.json")) 
                                  or os.path.isdir(path)):
        config.model_path = path
        break

if not config.model_path:
    # Last resort: try HuggingFace download (only works with internet ON)
    config.model_path = "Qwen/Qwen2.5-Math-7B-Instruct"
    print("WARNING: No local model found! Trying HuggingFace download...")
    print("If internet is OFF, this WILL fail.")
    print("Fix: Add Input -> Models -> search 'qwen2.5 math' -> add 7b-instruct")
else:
    print(f"Model found: {config.model_path}")

# List what's attached so you can debug path issues
print("\nAttached inputs:")
for p in glob.glob("/kaggle/input/*/"):
    print(f"  {p}")

In [ ]:
# =============================================
# PYTHON REPL - Safe code execution sandbox
# =============================================
# When the model writes ```python ... ```, we execute it and
# feed the output back. This is the "Tool" in Tool-Integrated Reasoning.
#
# Safety measures:
# - Runs in a temp directory (no file system access)
# - Timeout kills long-running code (infinite loops)
# - Blocked: subprocess, os.system, eval, exec, __import__

class PythonREPL:
    def __init__(self, timeout: int = 5):
        self.timeout = timeout
    
    @contextmanager
    def _time_limit(self, seconds):
        def handler(*_):
            raise TimeoutError(f"Code timed out after {seconds}s")
        old = signal.signal(signal.SIGALRM, handler)
        signal.alarm(seconds)
        try:
            yield
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old)
    
    def execute(self, code: str) -> tuple:
        """Execute Python code. Returns (success: bool, output: str)."""
        # Add common math imports
        preamble = (
            "import math\n"
            "import numpy as np\n"
            "import sympy as sp\n"
            "from sympy import *\n"
            "from fractions import Fraction\n"
            "from itertools import permutations, combinations, product\n"
            "from functools import reduce\n"
        )
        full_code = preamble + code
        
        # Make sure last line prints something
        lines = full_code.strip().split("\n")
        last = lines[-1].split("#")[0].strip()
        if last and "print(" not in last and not last.startswith(("import", "from", "#", "def", "class", "if", "for", "while", "try", "with")):
            lines[-1] = f"print({last})"
            full_code = "\n".join(lines)
        
        with tempfile.TemporaryDirectory() as tmpdir:
            path = os.path.join(tmpdir, "solution.py")
            with open(path, "w") as f:
                f.write(full_code)
            try:
                with self._time_limit(self.timeout):
                    result = subprocess.run(
                        ["python3", path],
                        capture_output=True, text=True, timeout=self.timeout,
                    )
                if result.returncode == 0:
                    return True, result.stdout.strip()[:1000]
                return False, result.stderr.strip()[-500:]
            except (TimeoutError, subprocess.TimeoutExpired):
                return False, "Timed out"
            except Exception as e:
                return False, str(e)[:200]

executor = PythonREPL(config.code_timeout)

# Test it
ok, out = executor.execute("print(sum(range(100)))")
print(f"Test: success={ok}, output={out}")

In [ ]:
# =============================================
# ANSWER EXTRACTION
# =============================================

def extract_boxed(text: str) -> str:
    """Extract answer from \\boxed{...} or \\fbox{...}."""
    for prefix in [r"\boxed", r"\fbox"]:
        idx = text.rfind(prefix)
        if idx < 0:
            continue
        i, depth = idx, 0
        while i < len(text):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    start = text.index("{", idx) + 1
                    return text[start:i].strip()
            i += 1
    return ""


def to_int(text: str) -> int:
    """Parse answer text to integer in range 0-99999."""
    if not text:
        return -1
    
    # Clean up common formatting
    for rm in [r"\text{", "}", r"\mathrm{", "$", ",", " ", "%",
               "square", "units", "degrees", "cm", "meters"]:
        text = text.replace(rm, "")
    
    # Handle fractions
    if "/" in text and text.replace("/", "").replace(".", "").replace("-", "").isdigit():
        try:
            from fractions import Fraction
            val = float(Fraction(text))
            return max(0, min(99999, round(val)))
        except Exception:
            pass
    
    try:
        val = round(float(text))
        return max(0, min(99999, val))
    except Exception:
        return -1


# Test
print(extract_boxed(r"The answer is \boxed{42}"))   # -> "42"
print(to_int("42"))                                  # -> 42
print(to_int("1,234"))                               # -> 1234

In [ ]:
# =============================================
# LOAD MODEL
# =============================================
# Two backends:
#   vLLM:         Fast batched inference via PagedAttention
#   transformers: Always available, generates one sample at a time

if USE_VLLM:
    # --- vLLM path (fast) ---
    llm = LLM(
        model=config.model_path,
        dtype="bfloat16",
        tensor_parallel_size=1,
        max_model_len=4096,
        trust_remote_code=True,
        gpu_memory_utilization=0.90,
    )
    tok = llm.get_tokenizer()
    print(f"Model loaded with vLLM: {config.model_path}")

else:
    # --- transformers path (no install needed) ---
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    print(f"Loading tokenizer from {config.model_path}...")
    tok = AutoTokenizer.from_pretrained(
        config.model_path,
        trust_remote_code=True,
    )
    # Left-padding is needed for batch generation
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    
    print(f"Loading model from {config.model_path}...")
    model_hf = AutoModelForCausalLM.from_pretrained(
        config.model_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model_hf.eval()
    print(f"Model loaded on {model_hf.device} with transformers")

In [ ]:
# =============================================
# SC-TIR: The Core Algorithm
# =============================================

BLOCKED_COMMANDS = ["subprocess", "os.system", "__import__", "shutil", "open(", "eval(", "exec("]

def extract_last_code_block(text: str) -> str:
    """Extract the last ```python ... ``` block from text."""
    blocks = re.findall(r"```python\s*(.*?)```", text, re.DOTALL)
    if not blocks:
        return ""
    code = blocks[-1].strip()
    for cmd in BLOCKED_COMMANDS:
        if cmd in code:
            return ""
    return code


def build_prompt(problem: str) -> str:
    """Build the chat prompt for a math problem."""
    messages = [{"role": "user", "content": (
        "Solve this math problem step by step. "
        "Use Python code when calculations are needed.\n"
        "Put your final numerical answer in \\boxed{}.\n\n"
        f"Problem: {problem}"
    )}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _generate_vllm(candidates: list[str]) -> list[str]:
    """Generate next step for all candidates using vLLM (fast, batched)."""
    sampling = SamplingParams(
        temperature=config.temperature,
        max_tokens=config.max_tokens,
        stop=["```output"],
        include_stop_str_in_output=True,
    )
    outputs = llm.generate(candidates, sampling)
    return [o.prompt + o.outputs[0].text for o in outputs]


def _generate_transformers(candidates: list[str]) -> list[str]:
    """Generate next step for all candidates using transformers (one at a time)."""
    results = []
    for text in candidates:
        inputs = tok(
            text, return_tensors="pt",
            truncation=True, max_length=3500
        ).to(model_hf.device)
        input_len = inputs["input_ids"].shape[1]
        
        with torch.no_grad():
            output_ids = model_hf.generate(
                **inputs,
                max_new_tokens=config.max_tokens,
                temperature=config.temperature,
                do_sample=True,
                pad_token_id=tok.eos_token_id,
            )
        
        new_text = tok.decode(output_ids[0][input_len:], skip_special_tokens=True)
        full_text = text + new_text
        
        # Simulate stop_strings: truncate at ```output if present
        stop_marker = "```output"
        stop_idx = new_text.find(stop_marker)
        if stop_idx >= 0:
            full_text = text + new_text[:stop_idx + len(stop_marker)]
        
        results.append(full_text)
    return results


def solve(problem: str) -> int:
    """
    Solve a math problem using SC-TIR.
    
    1. Create N copies of the prompt
    2. Generate solutions (stop at ```output to execute code)
    3. Execute code, append result, continue generating
    4. Repeat M times
    5. Extract answers, majority vote
    """
    prompt = build_prompt(problem)
    candidates = [prompt] * config.num_samples
    
    generate_fn = _generate_vllm if USE_VLLM else _generate_transformers
    
    # M rounds of generate -> execute -> continue
    for step in range(config.num_generations):
        candidates = generate_fn(candidates)
        
        # Execute code blocks where model stopped at ```output
        new_candidates = []
        for text in candidates:
            if text.rstrip().endswith("```output"):
                code = extract_last_code_block(text)
                if code:
                    success, result = executor.execute(code)
                    text += f"\n{result}\n```\n"
                else:
                    text += "\n\n```\n"
            new_candidates.append(text)
        candidates = new_candidates
    
    # Extract answers from all candidates
    answers = []
    for text in candidates:
        boxed = extract_boxed(text)
        if boxed:
            val = to_int(boxed)
            if val >= 0:
                answers.append(val)
    
    # Majority vote
    if not answers:
        return 0
    
    winner = Counter(answers).most_common(1)[0][0]
    print(f"  -> {len(answers)}/{config.num_samples} answers extracted, winner={winner}")
    return winner


# Quick test
print("Testing solve on a simple problem...")
test_answer = solve("What is 17 * 23?")
print(f"Result: 17*23 = {test_answer} (expected 391)")

In [ ]:
# =============================================
# SUBMISSION / LOCAL EVALUATION
# =============================================

if IS_SUBMISSION:
    # --- Kaggle Competition Mode ---
    # The evaluation API sends problems one at a time.
    # We solve each and return the answer.
    import kaggle_evaluation.aimo_3_inference_server
    
    def predict(problem_df: pd.DataFrame) -> pd.DataFrame:
        problem = problem_df["problem"].iloc[0]
        answer = solve(problem)
        return pd.DataFrame({"id": problem_df["id"], "answer": [answer]})
    
    server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
    server.serve()

else:
    # --- Local Testing Mode ---
    # Test on the 10 reference problems to verify everything works.
    ref_path = "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
    if not os.path.exists(ref_path):
        ref_path = "../data/reference.csv"  # Local fallback
    
    test_df = pd.read_csv(ref_path)
    print(f"Testing on {len(test_df)} reference problems...\n")
    
    correct = 0
    for idx, row in test_df.iterrows():
        pred = solve(row["problem"])
        true = row["answer"]
        is_ok = pred == true
        correct += is_ok
        print(f"#{idx+1}: pred={pred}, true={true} {'OK' if is_ok else 'WRONG'}")
    
    print(f"\nResult: {correct}/{len(test_df)} = {correct/len(test_df):.0%}")